# **Работа с API**

# **Получение информации обо всех книгах**

In [3]:
import requests
import pandas as pd

books_url = 'https://www.anapioficeandfire.com/api/books'
response = requests.get(books_url)
books_data = response.json()

df_books = pd.DataFrame(books_data)
print(f"Найдено {len(df_books)} книг.")
df_books.head()

Найдено 10 книг.


,url,name,isbn,authors,numberOfPages,publisher,country,mediaType,released,characters,povCharacters
0,https://www.anapioficeandfire.com/api/books/1,A Game of Thrones,978-0553103540,[George R. R. Martin],694,Bantam Books,United States,Hardcover,1996-08-01T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
1,https://www.anapioficeandfire.com/api/books/2,A Clash of Kings,978-0553108033,[George R. R. Martin],768,Bantam Books,United States,Hardback,1999-02-02T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
2,https://www.anapioficeandfire.com/api/books/3,A Storm of Swords,978-0553106633,[George R. R. Martin],992,Bantam Books,United States,Hardcover,2000-10-31T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
3,https://www.anapioficeandfire.com/api/books/4,The Hedge Knight,978-0976401100,[George R. R. Martin],164,Dabel Brothers Publishing,United States,GraphicNovel,2005-03-09T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
4,https://www.anapioficeandfire.com/api/books/5,A Feast for Crows,978-0553801507,[George R. R. Martin],784,Bantam Books,United Status,Hardcover,2005-11-08T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...


# **Получение информации обо всех домах Вестероса**

In [4]:
houses_url = 'https://www.anapioficeandfire.com/api/houses'
houses = []
page = 1

while True:
  url = f'https://www.anapioficeandfire.com/api/houses?page={page}&pageSize=50'
  response = requests.get(url)
  data = response.json()

  if not data:
    break

  houses.extend(data)
  page += 1

print(f'Загружено домов: {len(houses)}')

houses_df = pd.DataFrame(houses)
print(houses_df['region'].unique())


Загружено домов: 444
['The Westerlands' 'Dorne' 'The North' 'The Reach' 'The Vale'
 'The Riverlands' 'The Crownlands' 'The Stormlands' '' 'The Neck'
 'Iron Islands' 'Beyond the Wall' 'None']


In [5]:
westeros_regions = [
    'The Westerlands',
    'Dorne',
    'The North',
    'The Reach',
    'The Vale',
    'The Riverlands',
    'The Crownlands',
    'The Stormlands',
    'Iron Islands'
]

westeros_houses = houses_df[houses_df['region'].isin(westeros_regions)]
print(f'Найдено домов из Вестероса: {len(westeros_houses)}')

westeros_houses[['name', 'region', 'words']].head()

Найдено домов из Вестероса: 429


,name,region,words
0,House Algood,The Westerlands,
1,House Allyrion of Godsgrace,Dorne,No Foe May Pass
2,House Amber,The North,
3,House Ambrose,The Reach,Never Resting
4,House Appleton of Appleton,The Reach,


# **Получение информации обо всех домах Вестероса, у которых есть девиз**

In [7]:
houses_with_words = [house for house in houses if house['words']]

df_houses_with_words = pd.DataFrame(houses_with_words)
print(f'Найдено {len(df_houses_with_words)} домов с девизами.')
df_houses_with_words[['name', 'words']].head()

Найдено 68 домов с девизами.


,name,words
0,House Allyrion of Godsgrace,No Foe May Pass
1,House Ambrose,Never Resting
2,House Arryn of the Eyrie,As High as Honor
3,House Ashford of Ashford,Our Sun Shines Bright
4,House Baratheon of Storm's End,Ours is the Fury


# **Работа с DataBase**

# **Установка библиотеки psycopg2 и подключение к DataBase**

In [8]:
!pip install psycopg2-binary
import psycopg2

conn = psycopg2.connect(
    host='hh-pgsql-public.ebi.ac.uk',
    database='pfmegrnargs',
    user='reader',
    password='NWDMCE5xdipIjRrp'
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 22.2 MB/s eta 0:00:00


# **Получение первых 10 строк, сохранение в pd.DF**

In [11]:
cur = conn.cursor()

cur.execute("SELECT * FROM rnc_database LIMIT 10;")

rows = cur.fetchall()

cur.close()

df = pd.DataFrame(rows)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,1,2017-05-01 00:00:00.000000,RNACEN,ENA,884,ENA,Y,,ENA,,412.0,10.0,900074.0,12086180,814855,provides a comprehensive record of the world's...,https://www.ebi.ac.uk/ena/browser/,"[{'upi': 'URS00002D0E0C', 'taxid': 10090}, {'u...",[{'title': 'The European Nucleotide Archive in...
1,5,2017-05-17 00:00:00.000000,RNACEN,VEGA,98,VEGA,N,,VEGA,PRJEB4568,NaN,NaN,NaN,0,0,is a repository for high-quality gene models p...,http://vega.sanger.ac.uk/,"[{'upi': 'URS00000B15DA', 'taxid': 9606}, {'up...",[{'title': 'The GENCODE v7 catalog of human lo...
2,26,2017-05-01 00:00:00.000000,RNACEN,GENCODE,450,GENCODE,N,,GENCODE,,889.0,32.0,205012.0,47677,2,produces high quality reference gene annotatio...,http://gencodegenes.org/,"[{'upi': 'URS00000B15DA', 'taxid': 9606}, {'up...",[{'title': 'GENCODE: the reference human genom...
3,55,2023-10-10 15:02:45.191606,RNACEN,MGNIFY,839,MGnify,Y,None,MGnify,None,151.0,27.0,3514.0,135924,1929,None,None,None,None
4,41,2017-05-01 00:00:00.000000,RNACEN,GENECARDS,867,MalaCards,Y,,GeneCards,,1298.0,16.0,347561.0,517673,1,"is a searchable, integrative database that pro...",https://www.genecards.org/,"[{'upi': 'URS0000EBFCE3', 'taxid': 9606}, {'up...",[{'title': 'The GeneCards Suite: From Gene Dat...


# **Получение только нужных в задании столбцов**

In [12]:
cur = conn.cursor()

cur.execute("""
    SELECT display_name, num_sequences, num_organisms, url
    FROM rnc_database
    LIMIT 10;
""")

rows_selected = cur.fetchall()

cur.close()

df_selected = pd.DataFrame(rows_selected, columns=['display_name', 'num_sequences', 'num_organisms', 'url'])
df_selected.head()

,display_name,num_sequences,num_organisms,url
0,ENA,12086180,814855,https://www.ebi.ac.uk/ena/browser/
1,VEGA,0,0,http://vega.sanger.ac.uk/
2,GENCODE,47677,2,http://gencodegenes.org/
3,MGnify,135924,1929,None
4,GeneCards,517673,1,https://www.genecards.org/
